In [1]:
import pandas as pd
import altair as alt
import geopandas as gpd

from ecostyles import EcoStyles
# Create styles instance
styles = EcoStyles()
# Register and enable a theme
styles.register_and_enable_theme(theme_name="article")  # or "article"

In [2]:
world_tjosn_link = 'https://raw.githubusercontent.com/jhellingsdata/map-data/refs/heads/main/world/world.json'

In [10]:
alignment_path = '/Users/sambickel-barlow/RADataHub/RADataHub/Article Charts/latam_summary/latam_political_alignment.csv'
alignment_df = pd.read_csv(alignment_path)

alignment_df = pd.concat(
    [
        alignment_df,
        pd.DataFrame(
            [
                {'Year': 2022, 'Country': 'Guyana', 'Party': 'Left'},
                {'Year': 2026, 'Country': 'Guyana', 'Party': 'Left'},
                {'Year': 2022, 'Country': 'Suriname', 'Party': 'Left'},
                {'Year': 2026, 'Country': 'Suriname', 'Party': 'Left'},
                {'Year': 2022, 'Country': 'Jamaica', 'Party': 'Right'},
                {'Year': 2026, 'Country': 'Jamaica', 'Party': 'Right'},
                {'Year': 2022, 'Country': 'Puerto Rico', 'Party': 'Right'},
                {'Year': 2026, 'Country': 'Puerto Rico', 'Party': 'Right'},
            ]
        ),
    ],
    ignore_index=True,
)

party_scale = alt.Scale(domain=['Left', 'Right'], range=['#E6224B', '#36B7B4'])


def make_latam_map(year, title):
    year_df = alignment_df.loc[alignment_df['Year'] == year, ['Country', 'Party']].copy()

    return (
        alt.Chart(alt.topo_feature(world_tjosn_link, 'geog'))
        .transform_filter("datum.properties.region_wb == 'Latin America & Caribbean'")
        .transform_lookup(
            lookup='properties.name_long',
            from_=alt.LookupData(year_df, 'Country', ['Party'])
        )
        .mark_geoshape(stroke='white', strokeWidth=0.6)
        .encode(
            color=alt.Color(
                'Party:N',
                scale=party_scale,
                legend=alt.Legend(
                    title=None,
                    orient='bottom-left',
                    direction='vertical',
                    symbolType='square',
                    labelFont='Circular Std',
                    labelFontSize=16,
                    symbolSize=400,
                    padding=0,
                    offset=10,
                ),
            )
            ,
            tooltip=[
                alt.Tooltip('properties.name_long:N', title='Country'),
                alt.Tooltip('Party:N', title='Alignment'),
            ]
        )
        .project(type='naturalEarth1')
        .properties(width=320, height=420, title=title)
    )


latam_map = alt.hconcat(
    make_latam_map(2022, 'End of 2022'),
    make_latam_map(2026, 'End of 2026'),
    spacing=30,
).resolve_scale(color='shared').configure_view(stroke=None).configure_concat(spacing=30).configure_title(fontSize=16, anchor='middle')

latam_map

alt.HConcatChart(...)

In [11]:
# Save to png
latam_map.save('latam_map.png', scale_factor=2)
# Save to json
latam_map.save('latam_map.json', scale_factor=2)